In [1]:
from dotenv import load_dotenv,find_dotenv
load_dotenv(find_dotenv(),override=True)

True

In [2]:
from google import genai

In [3]:
import os
client=genai.Client(
    api_key=os.getenv("Gemini_API_Key")
)

In [4]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2


In [5]:
response=client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents='What is AI?'
)
print(response.text)

**AI** stands for **Artificial Intelligence**. At its broadest level, AI refers to the development of computer systems and machines that can perform tasks that typically require human intelligence. 

These tasks include learning, reasoning, problem-solving, perception, language understanding, and even creativity.

Here is a breakdown to help you understand AI better:

### 1. How Does It Work?
Traditionally, computers are programmed by humans to follow strict rules: *“If this happens, do that.”* 

AI, on the other hand, often uses **Machine Learning (ML)**. Instead of being given rigid rules, an AI system is fed massive amounts of data and uses algorithms to find patterns, learn from experience, and make decisions or predictions on its own. **Generative AI** (like ChatGPT) takes this a step further, allowing computers to create entirely new content, such as text, images, and music.

### 2. Types of AI
AI is generally categorized in two ways:
*   **By Capability:**
    *   **Narrow AI (W

# Load documents

In [6]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Home\AppData\Local\Temp\ipykernel_20612\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
data=PyPDFLoader('Artificial_Intelligence_10_Page_Document.pdf')
docs=data.load()

In [8]:
from langchain_core.prompts import ChatPromptTemplate

In [9]:
chat_prompt=ChatPromptTemplate([
    ("system","You are a helpful AI Assistant"),
    ("human","{data}")

    
])

In [10]:
prompt=chat_prompt.invoke({
    "data":docs[0].page_content
}).to_string()

print(type(prompt))

<class 'str'>


In [11]:
response=client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt
)
print(response.text)

It looks like you've shared the title page and introduction for a 10-page educational document about Artificial Intelligence. 

How can I help you with this document today? For example, I can:
* Summarize upcoming sections if you paste them.
* Answer specific questions about AI history, uses, benefits, or risks.
* Create study questions, flashcards, or quizzes based on the content.
* Help explain complex AI concepts in simpler terms. 

Please go ahead and share the next part of the text or let me know what you'd like to do!


# Text-Splitters

In [12]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [14]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
texts = text_splitter.split_documents(docs)

In [15]:
print(len(texts))

26


In [16]:
print(texts[0].page_content)

Artificial Intelligence | Educational Document
Page 1
 ARTIFICIAL
INTELLIGENCE
 An Introduction to Its History, Uses, Benefits, Risks, and Future
 A 10-Page Educational Document
 Prepared for general educational and study purposes


In [17]:
!pip install -U langchain-chroma langchain-google-genai

In [18]:
!pip install -U langchain-core langchain-google-genai langchain-chroma langchain-text-splitters langchain-community

In [19]:
import langchain_core
import langchain_google_genai
import langchain_chroma

print("LangChain Core:", langchain_core.__version__)
print("All packages imported successfully!")

LangChain Core: 1.8.0
All packages imported successfully!


In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

print("Imports Successful!")

Imports Successful!


In [21]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)



# Vector DB

In [22]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embeddings,
    persist_directory="chroma_langchain_db",
)

In [23]:
vector_store.add_documents(documents=texts)

print("PDF documents added successfully!")

PDF documents added successfully!


In [24]:
query = "What is Artificial Intelligence?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)


Document 1:
Artificial Intelligence | Educational Document
Page 2
Introduction to Artificial Intelligence
Overview
Artificial Intelligence (AI) is a branch of computer science concerned with creating machines and software that can
perform tasks commonly associated with human intelligence. These tasks include understanding language,
recognizing images, learning from experience, solving problems, making predictions, and supporting decisions. AI
is not a single machine or program. It is a broad collection of methods, technologies, and applications that allow
computer systems to respond intelligently to information.
Why AI Matters
AI has moved from research laboratories into everyday life. It helps people search the internet, receive
recommendations, translate languages, detect suspicious banking activity, navigate roads, and access digital
assistants. Businesses use AI to analyse large amounts of information and improve services. Governments and

Document 2:
Artificial Intelligence | Edu

In [25]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using only the following context.
If the answer is not available in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
""")

In [26]:
query = "What is Artificial Intelligence?"

results = vector_store.similarity_search(query, k=3)

context = "\n\n".join(
    doc.page_content for doc in results
)

prompt = rag_prompt.invoke({
    "context": context,
    "question": query
}).to_string()

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

Based on the provided context, Artificial Intelligence (AI) is a branch of computer science concerned with creating machines and software that can perform tasks commonly associated with human intelligence. It is not a single machine or program, but rather a broad collection of methods, technologies, and applications that allow computer systems to respond intelligently to information.
